In [ ]:
import pickle 
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import umap
import matplotlib.pyplot as plt
import cv2

In [ ]:
cluster_type = 'kmeans'

In [ ]:
#load for both radius=20 and radius=100
#cell_counts around cells 
#fluorescent marker intensity around cells
#fibrosis intensity around cells

#load pixels_outside_cells 
#load cells_subtype_clusters 
#load cluster_color_dict_map

In [ ]:
key = "" # sample identifier

In [ ]:
cell_clustering_path = r"" # path to cell phenotyping results

In [ ]:
cell_pixel_barcode_path = r"" # path to cell pixel barcode results

In [ ]:
cell_cluster_location_df = pd.read_csv(cell_clustering_path + '/cell_cluster_location_merged_unclassified_within_contour.csv')

In [ ]:
pixels_outside_cells = pd.read_csv(cell_pixel_barcode_path + '/radius100' + '/pixel_proportions_outside_cells.csv').drop(columns=['Unnamed: 0'])

In [ ]:
with open(cell_pixel_barcode_path + '/radius100' + '/cell_subtype_clusters_' + cluster_type + '.pickle', 'rb') as f: 
    cell_subtype_clusters = pickle.load(f)

In [ ]:
with open(cell_pixel_barcode_path + '/radius100' + '/cluster_color_dict_map_' + cluster_type + '.pickle', 'rb') as f: 
    cluster_color_dict_map = pickle.load(f)

In [ ]:
with open(cell_pixel_barcode_path + '/radius100' + '/image_labeled_dict_pixels_outside_cells_' + cluster_type + '.pickle', 'rb') as f: 
    image_labeled_dict = pickle.load(f)

In [ ]:
#radius=100

with open(cell_pixel_barcode_path + '/radius100' + '/cell_counts_outside_cells.pickle', 'rb') as f:
    cell_counts_outside_cells_radius100 = pickle.load(f)

with open(cell_pixel_barcode_path + '/radius100' + '/mean_fl_intensity_per_cell_outside_cells.pickle', 'rb') as f:
    fl_intensity_outside_cells_radius100 = pickle.load(f)

with open(cell_pixel_barcode_path + '/radius100' + '/fibrosis_intensity_counts_outside_cells.pickle', 'rb') as f:
    fibrosis_intensity_counts_outside_cells_radius100 = pickle.load(f)

with open(cell_pixel_barcode_path + '/radius100' + '/weighted_fibrosis_outside_cells.pickle', 'rb') as f:
    fibrosis_outside_cells_radius100 = pickle.load(f)

In [ ]:
cell_counts_outside_cells_radius100_df = pd.DataFrame(cell_counts_outside_cells_radius100)
cell_counts_outside_cells_radius100_df = cell_counts_outside_cells_radius100_df.fillna(0)
cell_proportions_outside_cells_radius100_df = cell_counts_outside_cells_radius100_df.div(cell_counts_outside_cells_radius100_df.sum(axis=1), axis=0)
cell_proportions_outside_cells_radius100_df['Cell ID'] = cell_cluster_location_df['Cell ID']
cell_proportions_outside_cells_radius100_df['X'] = cell_cluster_location_df['X']
cell_proportions_outside_cells_radius100_df['Y'] = cell_cluster_location_df['Y']
cell_proportions_outside_cells_radius100_df['Cell Type'] = cell_cluster_location_df['Cell Type']

cell_proportions_outside_cells_radius100_df = cell_proportions_outside_cells_radius100_df[cell_proportions_outside_cells_radius100_df['Cell ID'].isin(pixels_outside_cells['Cell ID'])]


fl_intensity_outside_cells_radius100_df = pd.DataFrame(fl_intensity_outside_cells_radius100)
fl_intensity_outside_cells_radius100_df['Cell ID'] = cell_cluster_location_df['Cell ID']
fl_intensity_outside_cells_radius100_df['X'] = cell_cluster_location_df['X']
fl_intensity_outside_cells_radius100_df['Y'] = cell_cluster_location_df['Y']
fl_intensity_outside_cells_radius100_df['Cell Type'] = cell_cluster_location_df['Cell Type']

fl_intensity_outside_cells_radius100_df = fl_intensity_outside_cells_radius100_df[fl_intensity_outside_cells_radius100_df['Cell ID'].isin(pixels_outside_cells['Cell ID'])]


fibrosis_intensity_counts_outside_cells_radius100_df = pd.DataFrame(fibrosis_intensity_counts_outside_cells_radius100, columns=['Fibrosis Intensity'])

fibrosis_intensity_counts_outside_cells_radius100_df['Cell ID'] = cell_cluster_location_df['Cell ID']
fibrosis_intensity_counts_outside_cells_radius100_df['X'] = cell_cluster_location_df['X']
fibrosis_intensity_counts_outside_cells_radius100_df['Y'] = cell_cluster_location_df['Y']
fibrosis_intensity_counts_outside_cells_radius100_df['Cell Type'] = cell_cluster_location_df['Cell Type']

fibrosis_intensity_counts_outside_cells_radius100_df = fibrosis_intensity_counts_outside_cells_radius100_df[fibrosis_intensity_counts_outside_cells_radius100_df['Cell ID'].isin(pixels_outside_cells['Cell ID'])]

fibrosis_outside_cells_radius100_df = pd.DataFrame(fibrosis_outside_cells_radius100, columns=['Fibrosis Fraction'])

fibrosis_outside_cells_radius100_df['Cell ID'] = cell_cluster_location_df['Cell ID']
fibrosis_outside_cells_radius100_df['X'] = cell_cluster_location_df['X']
fibrosis_outside_cells_radius100_df['Y'] = cell_cluster_location_df['Y']
fibrosis_outside_cells_radius100_df['Cell Type'] = cell_cluster_location_df['Cell Type']

fibrosis_outside_cells_radius100_df = fibrosis_outside_cells_radius100_df[fibrosis_outside_cells_radius100_df['Cell ID'].isin(pixels_outside_cells['Cell ID'])]


cell_environment_radius100_df = pd.concat([cell_proportions_outside_cells_radius100_df.iloc[:, :5], fibrosis_outside_cells_radius100_df.iloc[:, :1], fibrosis_intensity_counts_outside_cells_radius100_df], axis=1)   


In [ ]:
cell_environment_radius_df = cell_environment_radius100_df.fillna(0)

In [ ]:
cell_environment_radius = np.array(cell_environment_radius_df.iloc[:, 1:-3])

In [ ]:
cell_types = cell_environment_radius_df['Cell Type'].unique()
for cell_type in cell_types:
    if(cell_type !='Unclassified'):
        cell_environment_radius_df.loc[cell_environment_radius_df['Cell Type'] == cell_type, 'Barcode Cluster'] = cell_subtype_clusters[cell_type]

In [ ]:
cell_type_list = list(cell_environment_radius_df['Cell Type'])
target_cell_type = 'CD4+ T-cells'
colors = []
i = 0
for t in tqdm(cell_type_list):
    if(t == target_cell_type):
        id = list(cell_environment_radius_df['Cell ID'])[i] 
        cluster = int(cell_environment_radius_df[cell_environment_radius_df['Cell ID'] == id]['Barcode Cluster'].iloc[0])
        colors.append(cluster_color_dict_map[t][cluster])
    else:
        colors.append([0.9, 0.9, 0.9])
    i+=1 

In [ ]:
scaler = StandardScaler()
radius_scaled = scaler.fit_transform(cell_environment_radius)

umap_model = umap.UMAP(n_components=2, random_state=42) 
umap_radius = umap_model.fit_transform(radius_scaled)

In [ ]:
fig, ax = plt.subplots(1)
ax.scatter(umap_radius[:, 0], umap_radius[:, 1], s=1, c=colors)
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')

### Project within cell phenotype

In [ ]:
cell_type_list = list(cell_environment_radius_df['Cell Type'])
target_cell_type = 'CD8+ T-cells'
target_colors = []
i = 0
for t in tqdm(cell_type_list):
    if(t == target_cell_type):
        id = list(cell_environment_radius_df['Cell ID'])[i] 
        cluster = int(cell_environment_radius_df[cell_environment_radius_df['Cell ID'] == id]['Barcode Cluster'].iloc[0])
        target_colors.append(cluster_color_dict_map[t][cluster])

    i+=1 

In [ ]:
target_cell_environment_radius_df = cell_environment_radius_df[cell_environment_radius_df['Cell Type'] == target_cell_type]

In [ ]:
target_cell_environment_radius = np.array(target_cell_environment_radius_df.iloc[:, 1:-3])

In [ ]:
scaler = StandardScaler()
target_radius_scaled = scaler.fit_transform(target_cell_environment_radius)

In [ ]:
target_cell_environment_radius_scaled_df = pd.DataFrame(target_radius_scaled, columns=list(target_cell_environment_radius_df.columns[1:-3]))

In [ ]:
umap_model = umap.UMAP(n_components=2, random_state=42) 
target_umap_radius = umap_model.fit_transform(target_radius_scaled)

In [ ]:
import matplotlib as mpl

In [ ]:
mpl.rcParams["font.size"] = 20
fig, ax = plt.subplots(1)
fig.set_figheight(5)
fig.set_figwidth(8)
ax.scatter(target_umap_radius[:, 0], target_umap_radius[:, 1], s=4, c=target_colors)
plt.xlabel('UMAP 1')
plt.ylabel('UMAP 2')

In [ ]:
features = ['CD4+ T-cells', 'CD8+ T-cells', 'B-cells', 'Fibrosis Intensity', 'Fibrosis Fraction']

In [ ]:
title_features = ['Neighboring CD4+ T-cells', 'Neighboring CD8+ T-cells', 'Neighboring B-cells', 'Fibrosis Intensity', 'Fibrosis Fraction']

In [ ]:
mpl.rcParams["font.size"] = 20
fig, ax = plt.subplots(1, len(features))
fig.set_figheight(5)
fig.set_figwidth(50)
i = 0
for feature in features:
    scatter = ax[i].scatter(target_umap_radius[:, 0], target_umap_radius[:, 1], s=4, c=target_cell_environment_radius_scaled_df[feature], cmap='viridis')
    ax[i].set_xlabel('UMAP 1')
    ax[i].set_ylabel('UMAP 2')
    ax[i].set_title(title_features[i])
    fig.colorbar(scatter, ax=ax[i], orientation='vertical')
    i+=1 

In [ ]:
group_proportions = {}

for group, clusters in cell_subtype_clusters.items():
    if(group == 'Podocyte'):
        continue 
    unique_clusters, counts = np.unique(clusters, return_counts=True)
    total_count = len(clusters)
    proportions = {cluster: count / total_count for cluster, count in zip(unique_clusters, counts)}
    group_proportions[group] = proportions

fig, ax = plt.subplots(1)
fig.set_figheight(8)
fig.set_figwidth(8)

group_names = list(group_proportions.keys())
num_groups = len(group_names)

bar_width = 0.2

x_positions = np.arange(0, num_groups/2, 0.5)
bottom = np.zeros(num_groups)

for cluster in set([cluster for clusters in cell_subtype_clusters.values() for cluster in clusters]):
    cluster_heights = []
    for group in group_names:
        proportion = group_proportions[group].get(cluster, 0)
        cluster_heights.append(proportion)

    ax.bar(x_positions, cluster_heights, bottom=bottom, label=cluster, 
           color=[cluster_color_dict_map[group].get(cluster, 'gray') for group in group_names], width=bar_width)
    bottom += cluster_heights 

ax.set_ylabel('Fraction')
ax.set_xticks(x_positions) 
ax.set_xticklabels(group_names)
plt.subplots_adjust(bottom=0.2)

plt.show()



In [ ]:
group_proportions